In [1]:
import torch
import sys
sys.path.append("examples")

from toy_transformer import ToyTransformer
from modellens import ModelLens

model = ToyTransformer()
input_ids = torch.randint(0, 100, (1, 10))

lens = ModelLens(model)
print(lens)
print(f"Backend: {lens.adapter.type_of_adapter}")
print(f"\nLayers:")
for name in lens.layer_names():
    print(f"  {name}")

ModelLens(backend=pytorch, hooks=0, params=162,980)
Backend: pytorch

Layers:
  embed
  blocks
  blocks.0
  blocks.0.ln_1
  blocks.0.attn
  blocks.0.attn.out_proj
  blocks.0.ln_2
  blocks.0.mlp
  blocks.0.mlp.0
  blocks.0.mlp.1
  blocks.0.mlp.2
  blocks.1
  blocks.1.ln_1
  blocks.1.attn
  blocks.1.attn.out_proj
  blocks.1.ln_2
  blocks.1.mlp
  blocks.1.mlp.0
  blocks.1.mlp.1
  blocks.1.mlp.2
  blocks.2
  blocks.2.ln_1
  blocks.2.attn
  blocks.2.attn.out_proj
  blocks.2.ln_2
  blocks.2.mlp
  blocks.2.mlp.0
  blocks.2.mlp.1
  blocks.2.mlp.2
  ln_f
  lm_head


In [2]:
# 1. Hooks — basic activation capture
lens.attach_all()
output = lens.run(input_ids)
print(f"Output shape: {output.shape}")
print(f"Activations captured: {len(lens.get_activations())}")
print(f"Shapes:")
for name, shape in lens.hooks.get_shapes().items():
    print(f"  {name}: {shape}")

Output shape: torch.Size([1, 10, 100])
Activations captured: 27
Shapes:
  embed: torch.Size([1, 10, 64])
  blocks.0.ln_1: torch.Size([1, 10, 64])
  blocks.0.attn: torch.Size([1, 10, 64])
  blocks.0.ln_2: torch.Size([1, 10, 64])
  blocks.0.mlp.0: torch.Size([1, 10, 256])
  blocks.0.mlp.1: torch.Size([1, 10, 256])
  blocks.0.mlp.2: torch.Size([1, 10, 64])
  blocks.0.mlp: torch.Size([1, 10, 64])
  blocks.0: torch.Size([1, 10, 64])
  blocks.1.ln_1: torch.Size([1, 10, 64])
  blocks.1.attn: torch.Size([1, 10, 64])
  blocks.1.ln_2: torch.Size([1, 10, 64])
  blocks.1.mlp.0: torch.Size([1, 10, 256])
  blocks.1.mlp.1: torch.Size([1, 10, 256])
  blocks.1.mlp.2: torch.Size([1, 10, 64])
  blocks.1.mlp: torch.Size([1, 10, 64])
  blocks.1: torch.Size([1, 10, 64])
  blocks.2.ln_1: torch.Size([1, 10, 64])
  blocks.2.attn: torch.Size([1, 10, 64])
  blocks.2.ln_2: torch.Size([1, 10, 64])
  blocks.2.mlp.0: torch.Size([1, 10, 256])
  blocks.2.mlp.1: torch.Size([1, 10, 256])
  blocks.2.mlp.2: torch.Size([1,

In [3]:
from modellens.analysis.logit_lens import run_logit_lens, decode_logit_lens

# Create a simple vocab dict for decoding (0-99)
vocab = {i: str(i) for i in range(100)}
results = run_logit_lens(lens, input_ids, top_k=5)
decoded = decode_logit_lens(results, vocab=vocab)

for layer_name, predictions in list(decoded.items())[:5]:
    tokens_str = ", ".join(f"{tok} ({prob:.3f})" for tok, prob in predictions)
    print(f"{layer_name:30s} -> {tokens_str}")

embed                          -> 47 (0.065), 89 (0.031), 65 (0.024), 81 (0.020), 36 (0.020)
blocks.0.ln_1                  -> 47 (0.062), 65 (0.030), 89 (0.022), 27 (0.020), 36 (0.019)
blocks.0.attn                  -> 25 (0.012), 2 (0.012), 34 (0.012), 73 (0.012), 54 (0.012)
blocks.0.ln_2                  -> 47 (0.057), 65 (0.028), 89 (0.022), 27 (0.021), 51 (0.019)
blocks.0.mlp.2                 -> 28 (0.012), 16 (0.012), 4 (0.012), 35 (0.012), 92 (0.012)


In [4]:
# Residual stream
from modellens.analysis.residual_stream import run_residual_analysis, identify_critical_layers

block_layers = ["blocks.0", "blocks.1", "blocks.2"]
lens.attach_layers(block_layers)
residual_results = run_residual_analysis(lens, input_ids, layer_names=block_layers)

for name, data in residual_results["contributions"].items():
    print(f"{name:15s} | rel: {data['relative_contribution']:.3f} | cos: {data['cosine_similarity']:.3f}")

blocks.1        | rel: 0.280 | cos: 0.959
blocks.2        | rel: 0.264 | cos: 0.964


In [5]:
# Embeddings
from modellens.analysis.embeddings import run_embeddings_analysis

embed_results = run_embeddings_analysis(lens, input_ids)
print(f"\nEmbedding dim: {embed_results['embed_dim']}")
print(f"Sequence length: {embed_results['seq_length']}")
print(f"Similarity matrix shape: {embed_results['similarity_matrix'].shape}")


Embedding dim: 64
Sequence length: 10
Similarity matrix shape: torch.Size([10, 10])


In [6]:
from modellens.analysis.attention import run_attention_analysis, head_summary

attn_results = run_attention_analysis(lens, input_ids)
print(f"Number of attention layers: {attn_results['num_layers']}")

for name, data in attn_results["attention_maps"].items():
    print(f"{name}: heads={data['num_heads']}, seq={data['seq_length']}")

Number of attention layers: 3
blocks.0.attn: heads=10, seq=10
blocks.1.attn: heads=10, seq=10
blocks.2.attn: heads=10, seq=10


In [7]:
summary = head_summary(attn_results)
for name, data in list(summary.items())[:5]:
    print(f"\n{name}:")
    print(f"  Entropy:       {[f'{e:.2f}' for e in data['entropy']]}")
    print(f"  Max attention: {[f'{a:.2f}' for a in data['max_attention']]}")


blocks.0.attn:
  Entropy:       ['2.30', '2.28', '2.26', '2.29', '2.30', '2.29', '2.29', '2.28', '2.28', '2.28']
  Max attention: ['0.12', '0.14', '0.15', '0.13', '0.13', '0.12', '0.12', '0.14', '0.14', '0.15']

blocks.1.attn:
  Entropy:       ['2.23', '2.29', '2.28', '2.27', '2.27', '2.29', '2.28', '2.29', '2.29', '2.29']
  Max attention: ['0.20', '0.12', '0.14', '0.17', '0.13', '0.13', '0.14', '0.12', '0.12', '0.13']

blocks.2.attn:
  Entropy:       ['2.26', '2.27', '2.27', '2.28', '2.28', '2.28', '2.26', '2.27', '2.27', '2.28']
  Max attention: ['0.14', '0.16', '0.16', '0.15', '0.13', '0.13', '0.16', '0.16', '0.16', '0.13']


In [8]:
from modellens.analysis.activation_patching import run_activation_patching

# Clear any leftover hooks
for name, module in model.named_modules():
    module._forward_hooks.clear()

clean_input = torch.randint(0, 100, (1, 10))
corrupted_input = torch.randint(0, 100, (1, 10))
patch_results = run_activation_patching(lens, clean_input, corrupted_input)

print(f"Clean metric:     {patch_results['clean_metric']:.4f}")
print(f"Corrupted metric: {patch_results['corrupted_metric']:.4f}")
print(f"Total effect:     {patch_results['total_effect']:.4f}\n")

for layer, data in patch_results["patch_effects"].items():
    print(f"{layer:25s} | patched: {data['patched_metric']:.4f} | effect: {data['normalized_effect']:+.3f}")

Clean metric:     1.5139
Corrupted metric: 1.4701
Total effect:     -0.0438

blocks.0.attn             | patched: 1.3922 | effect: +2.780
blocks.0.mlp              | patched: 1.6462 | effect: -3.025
blocks.1.attn             | patched: 1.4241 | effect: +2.051
blocks.1.mlp              | patched: 1.7467 | effect: -5.320
blocks.2.attn             | patched: 1.3939 | effect: +2.741
blocks.2.mlp              | patched: 1.4556 | effect: +1.332
